In [0]:
from pyspark.sql.functions import current_timestamp, col

raw_path = "/Volumes/citibike_lakehouse/bronze/bronze_raw_data/"
target_table = "citibike_lakehouse.bronze.trips_raw"

df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "PERMISSIVE")
    .load(raw_path)
)

display(df_raw)

df_bronze = (
    df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_name"))
)

display(df_bronze.limit(10))
df_bronze.printSchema()

row_count = df_bronze.count()
print(f"Rows loaded: {row_count}")

if row_count == 0:
  raise Exception("No rows loaded")

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(target_table)
)

print(f"Bronze table written: {target_table}")

In [0]:
%sql
select * from citibike_lakehouse.bronze.trips_raw;

In [0]:
%sql
DESCRIBE HISTORY citibike_lakehouse.bronze.trips_raw;